In [ ]:
# Helper to point to project root if imports cause issues
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

## Prerequisites

This tutorial expects a **graph data directory** with the layout described below.
The dataset used in the accompanying paper is not distributed with this repository
(see the Data section of the README).

Expected layout:

```
<graph_dir>/
  instances/           # graph instance files (input features)
  max_cuts/            # max/min cut values, needed for approximation ratios
  fixed_angle_cuts/    # optional: fixed-angle baseline results
  results_p<P>/        # per-depth QAOA optimization results, incl. best_param.json
  splits/              # train, validation and test splits
```

The datamodule works with any data source following this structure.

We define the path to the data directory and the QAOA depth once at the top of the notebook:


In [ ]:
from pathlib import Path

# ---- CHANGE PATH HERE! ------
GRAPH_DIR = Path("<path to your graph_dir>")

# QAOA depth used throughout the notebook
P = 3

print(f"Resolved path: {GRAPH_DIR.resolve()}")

## What input data is needed and used?

### 0. Instance Directory and Best-Param File

The datamodule needs two things:

1. **An instance directory**  
   A directory like `<graph_dir>/instances/` that contains one subfolder per instance set with one JSON file per graph (`<instance_set>/<instance_id>.json`), each describing a single instance (nodes, edges, weights).

2. **A best-param (summary) file**  
   A condensed JSON file summarizing a `results_p<P>/` folder (e.g. `<graph_dir>/results_p3/best_param.json`).  
   It contains the optimized QAOA parameters (training targets), keyed by instance set and instance id, and the name of the result file each entry came from (kept for traceability).  
   Together, **instance directory + best-param file** define the full input/target dataset.

In principle, adding more instances to a best-param file does not change training **if** fixed split files are used.  
However, different optimized parameters *do* change the dataset and make runs non-comparable.

See step 3 of the README's Data section for how to build
`results_p<P>/best_param.json` from raw optimization results.

In [ ]:
BEST_PARAM_FILE = GRAPH_DIR / f"results_p{P}" / "best_param.json"

print(f"Using best-param file: {BEST_PARAM_FILE.resolve()}")
# If this doesn't exist yet, build it first -- see step 3 of the
# README's Data section.

We can have a look at the entry of one instance, keyed by depth:

```json
"2": {
  "energy": 38.755814325501206,
  "qaoa_angles": [
    0.5563398645429792,
    0.29913147766071224,
    0.4917578063069325,
    0.893889795420136
  ],
  "result_file_name": "<result>.json"
}

In [ ]:
import json

# Load best-param file
with open(BEST_PARAM_FILE, "r") as f:
    data = json.load(f)

# data is keyed by instance_set, then instance_id, then depth p
first_set, by_instance = next(iter(data.items()))
first_instance, entry = next(iter(by_instance.items()))
print(f"{first_set}/{first_instance}:")
print(json.dumps(entry, indent=2))

### 1. Generate test split file

Before training, we hold out a fixed test set so it never influences training or
normalization statistics. `scripts_and_notebooks/testset_generation.py` builds this
master split (and the train/val split, plus `norm_stats`) directly from the instance
directory. This does not need a best-param file, so the same split can be reused
across every QAOA depth `p` you train on.

Typically you'd run this once from the command line:

```bash
python -m scripts_and_notebooks.testset_generation \
    --graphs_repo <graph_dir> --instance_sets random_regular erdos_renyi
```

Here we call the underlying function directly so we can inspect the result inline.

In [ ]:
from scripts_and_notebooks.testset_generation import generate_test_indices_file

# Written to a separate folder so the master splits from testset_generation.py are not overwritten.
TEST_SPLIT_FILE = GRAPH_DIR / "splits" / "tutorial" / "test_split.json"

test_ids, out_path = generate_test_indices_file(
    instance_dir  = GRAPH_DIR / "instances",
    out_path      = TEST_SPLIT_FILE,
    instance_set  = ["random_regular"],
    test_fraction = 0.15,
    seed          = 42,
)

In [ ]:
import json
from pathlib import Path

out_path = Path(out_path)
with out_path.open("r") as f:
    data = json.load(f)

# show first few test indices
print("First 4 test indices:", data["test_indices"][:4])

### 2. Datamodule

We now have everything we need for the datamodule: an **instance directory** for input features, a **best-param file** for target features, and a **test split file** for a hold-out set.

The datamodule builds on a PyTorch `Dataset` and a Lightning `DataModule` to provide ML-friendly train/val/test `DataLoader`s. It reads a best-param file and filters samples by `target_p` and a list of `instance_set`s. It returns:

- **x (inputs):** Dict of selected `in_features` (e.g., node_count, num_edges, mean_degree, std_degree).  
- **y (targets):** Dict of selected `out_features` (e.g., qaoa_params, opt_energy, optional opt_max_cut / opt_min_cut / opt_approx_ratio).

#### Required configs
- `summary_file`: best-param file  
- `instance_dir`: path to instance directory  
- `target_p`: depth p; results for this depth must be present in the best-param file  
- `instance_set`: list of instance directories to load from (defaults to `["random_regular"]`)  
- `train_fraction`: proportion of samples used for training (e.g., 0.8, rest is validation). The split is deterministic given the seed. You can also save and later reload a split file.
- `test_file`: path to a test split file; all listed indices are excluded from both train and val

#### Optional configs
- `in_features`: keys used for **x** (defaults to `BASIC_IN_FEATURES` in `datamodule/meta_configs.py`)  
- `out_features`: keys used for **y** (e.g., `qaoa_params`, `opt_energy`, `opt_max_cut`, …)  
- `compute_cut_metrics`: optionally compute max-cut metrics on the fly (defaults to `False`)  
- `normalize`: normalize features based on train-set statistics (computed on the fly). `SCALAR_FEATURES_IN`/`SCALAR_FEATURES_OUT` in `datamodule/meta_configs.py` define what is normalized. Target parameters are linearly scaled to `[0, 1]` if `normalize=True`.  
- `norm_stats`: optionally pass precomputed mean/std values  
- `cut_metrics_dir`: directory with max-cut results (the `max_cuts/` directory described in the readme). This speeds up validation and testing because optimal max-cuts are fetched instead of recomputed.
- Other standard configs: `batch_size`, `num_workers`, etc.

In practice, a bullet-proof training setup uses both a predefined train/val split file and precomputed normalization stats. We will do both below and show how to use them in a training run.

In [ ]:
from datamodule.datamodule import QAOADataModule
from datamodule.meta_configs import BASIC_IN_FEATURES

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
datamodule = QAOADataModule(
    summary_file    = BEST_PARAM_FILE,
    instance_dir    = GRAPH_DIR / "instances",
    test_file       = TEST_SPLIT_FILE,
    cut_metrics_dir = GRAPH_DIR / "max_cuts",

    target_p     = P,
    instance_set = ["random_regular"],
    train_fraction = 0.85,
    in_features  = BASIC_IN_FEATURES + ["node_count", "edges"],
)

In [ ]:
datamodule.setup()

In [ ]:
# Check len of train and val dataset
train_dataset = datamodule.train_dataset
val_dataset = datamodule.val_dataset
test_dataset = datamodule.test_dataset

print(len(train_dataset))
print(len(val_dataset)) # Val Set slightly smaller given we take 15% of the data already excluding test ids
print(len(test_dataset))

In [ ]:
# Inspect scalar x in_features
for i in range(0, 2):
    x, y = train_dataset[i]
    print(x.keys())
    print(y.keys())
    print()
    print(f"Node Count: {x['node_count']}")
    print(f"Mean Degree: {x['mean_degree']}")
    print(f"Num Edges: {x['num_edges']}")
    print()
    print(f"Target params: {y['qaoa_params']}")
    print(f"Opt Approximation Ratio: {y['opt_approx_ratio']}")
    print()

In [ ]:
## Edges are padded to max edge size
## Note, for now train/val and test data all needs to come from the same best param file such that the max_edge count can be globally computed and equally applied to all data.
print(x["edges"])

### Normalization

There are two ways to use normalization:

1. Set `normalize=True` (default) and do not pass `norm_stats`. The dataset is split into train/val and normalization statistics are computed on the training split during `.setup()`. For larger datasets this can take a bit of time once per setup.

2. Pass `norm_stats` (for all scalar features and `y`) to apply precomputed normalization statistics.

In [ ]:
# Option 1
datamodule = QAOADataModule(
    summary_file    = BEST_PARAM_FILE,
    instance_dir    = GRAPH_DIR / "instances",
    test_file       = TEST_SPLIT_FILE,
    cut_metrics_dir = GRAPH_DIR / "max_cuts",

    target_p     = P,
    instance_set = ["random_regular"],
    normalize    = True,
)

In [ ]:
datamodule.setup()
train_dataset = datamodule.train_dataset
val_dataset = datamodule.val_dataset

In [ ]:
for i in range(0, 3):
    x, y = train_dataset[i]
    print(f"Num Nodes: {x['num_nodes']}")

In [ ]:
# We can retrieve the (mean, std) tuples for all input features that are normalized
raw_stats = datamodule.norm_stats

clean_stats = {
    split: {
        k: (round(float(v[0]), 2), round(float(v[1]), 2))
        for k, v in feats.items()
    }
    for split, feats in raw_stats.items()
}

print(clean_stats)


In [ ]:
# Quick Sanity Check: Lets rebuild our datamodule with the norm stats as input

# Option 2
datamodule = QAOADataModule(
    summary_file    = BEST_PARAM_FILE,
    instance_dir    = GRAPH_DIR / "instances",
    test_file       = TEST_SPLIT_FILE,
    cut_metrics_dir = GRAPH_DIR / "max_cuts",
    target_p       = P,
    train_fraction = 0.85,
    instance_set   = ["random_regular"],
    # Values below are illustrative only -- use the stats printed above for your own data.
    norm_stats      = {'x': 
                       {'num_nodes': (61.36, 25.86), 
                        'num_edges': (193.76, 129.66), 
                        'edges_per_node': (2.95, 1.23), 
                        'mean_degree': (5.9, 2.45), 
                        'std_degree': (1.01, 1.09)}
                      }
)

In [ ]:
datamodule.setup()
train_dataset = datamodule.train_dataset
val_dataset = datamodule.val_dataset

In [ ]:
# As expected we only see slight deviations due to rounding of the norm stats above.
for i in range(0, 3):
    x, y = train_dataset[i]
    print(f"Num Nodes: {x['num_nodes']}")

### Train-Val Split

In principle the train val split use a seprate deterministic seed (you can control it by train_val_split_seed) and is therefore determinisitc given the same input data size and same train_fraction chosen, however to be one the save side we can save a train-val split and reuse this across trainings

In [ ]:
# One Time Generation of Train-Val Split
SPLIT_FILE = GRAPH_DIR / "splits" / "tutorial" / "train_val_split.json"

datamodule = QAOADataModule(
    summary_file    = BEST_PARAM_FILE,
    instance_dir    = GRAPH_DIR / "instances",
    test_file       = TEST_SPLIT_FILE,
    cut_metrics_dir = GRAPH_DIR / "max_cuts",

    target_p     = P,
    instance_set = ["random_regular"],
    save_split   = True,
    split_file   = SPLIT_FILE,
)

In [ ]:
datamodule.setup()

In [ ]:
# Using the split file
datamodule = QAOADataModule(
    summary_file    = BEST_PARAM_FILE,
    instance_dir    = GRAPH_DIR / "instances",
    test_file       = TEST_SPLIT_FILE,
    cut_metrics_dir = GRAPH_DIR / "max_cuts",

    target_p     = P,
    instance_set = ["random_regular"],
    split_file   = SPLIT_FILE,
)

In [ ]:
datamodule.setup()

## Summary / Best Practices for Training

1. **Fix the best-param file.**  
   Use the same best-param file as the underlying data source for both train/val and test splits, and do not change it across comparison runs. Create this file once at the beginning and include all data you might need later (e.g., if you want multi-graph comparisons, include all graph types and filter later in the datamodule).

2. **Fix the test split once.**  
   Generate a one-time test-indices file and do not touch it. As above, create it for the maximum set of graph types you want to cover and later sub-filter in the datamodule. This ensures the same data is always excluded across all evaluations.  
   Ideally, points 1 and 2 are standardized once and version-controlled alongside the dataset, so every run and every collaborator uses identical splits.

3. **Reuse train/val splits and norm stats.**  
   Configure your datamodule once to save a train/val split and compute normalization statistics, then reuse both across training and testing runs.